# Reportes Regulatorios CNSF

## Generación Automatizada de Reportes Trimestrales

La CNSF requiere que las aseguradoras presenten reportes trimestrales sobre suscripción, siniestros, inversiones y capital. Este notebook demuestra la generación automatizada de estos reportes.

### Contenido
1. Marco regulatorio de reportes CNSF
2. Reporte de Suscripción (primas y pólizas)
3. Reporte de Siniestros (siniestralidad y reservas)
4. Reporte de Inversiones (portafolio y riesgo)
5. Reporte RCS (capital regulatorio)
6. Exportación a Excel y PDF

In [ ]:
# Imports necesarios
from decimal import Decimal
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, date

# Imports de reportes
from mexican_insurance.reportes import (
    GeneradorSuscripcion,
    GeneradorSiniestros,
    GeneradorInversiones,
    GeneradorRCS,
    ExportadorExcel
)

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Configuración de pandas para visualización
pd.options.display.float_format = '{:,.2f}'.format

print("✅ Imports completados exitosamente")

## 1. Marco Regulatorio de Reportes CNSF

### Reportes Trimestrales Requeridos:

1. **Estados Financieros**
   - Balance General
   - Estado de Resultados
   - Flujo de Efectivo

2. **Información Estadística**
   - Suscripción (primas emitidas, en vigor)
   - Siniestros (pagados, pendientes, IBNR)
   - Reservas técnicas

3. **Información de Inversiones**
   - Portafolio de inversiones
   - Calificaciones y riesgo
   - Límites regulatorios

4. **Capital y Solvencia**
   - RCS (Requerimiento de Capital de Solvencia)
   - Capital disponible
   - Ratio de solvencia

### Plazos de Presentación:
- Trimestrales: 30 días después del cierre del trimestre
- Anuales: 60 días después del cierre del ejercicio

## 2. Reporte de Suscripción (Primas y Pólizas)

Generemos el reporte trimestral de suscripción.

In [ ]:
# Cargar datos de cartera
cartera_df = pd.read_csv('../examples/data/cartera_ejemplo.csv')

print("📊 REPORTE TRIMESTRAL DE SUSCRIPCIÓN")
print("="*90)
print(f"Período: Q3 2024 (Julio - Septiembre)")
print(f"Fecha de generación: {datetime.now().strftime('%Y-%m-%d')}")
print("="*90)

# Generar reporte
generador_susc = GeneradorSuscripcion()
reporte_susc = generador_susc.generar(
    cartera=cartera_df,
    periodo_inicio=date(2024, 7, 1),
    periodo_fin=date(2024, 9, 30)
)

# 1. Primas por Línea de Negocio
print(f"\n1. PRIMAS POR LÍNEA DE NEGOCIO")
print(f"   {'─'*80}")

primas_linea = reporte_susc['primas_por_linea']
print(primas_linea.to_string(index=False))

# 2. Pólizas en Vigor
print(f"\n2. PÓLIZAS EN VIGOR")
print(f"   {'─'*80}")
print(f"   Pólizas al inicio del trimestre:  {reporte_susc['polizas_inicio']:>10,}")
print(f"   Pólizas emitidas:                 {reporte_susc['polizas_emitidas']:>10,}")
print(f"   Pólizas canceladas:               {reporte_susc['polizas_canceladas']:>10,}")
print(f"   Pólizas al final del trimestre:   {reporte_susc['polizas_fin']:>10,}")

# 3. Resumen de Primas
print(f"\n3. RESUMEN DE PRIMAS (Miles de MXN)")
print(f"   {'─'*80}")
print(f"   Prima Emitida:                    ${reporte_susc['prima_emitida']/1000:>15,.0f}")
print(f"   Prima Devengada:                  ${reporte_susc['prima_devengada']/1000:>15,.0f}")
print(f"   Prima Cedida (Reaseguro):         ${reporte_susc['prima_cedida']/1000:>15,.0f}")
print(f"   Prima Retenida:                   ${reporte_susc['prima_retenida']/1000:>15,.0f}")

# 4. Indicadores
print(f"\n4. INDICADORES DE NEGOCIO")
print(f"   {'─'*80}")
print(f"   Tasa de retención de pólizas:     {reporte_susc['tasa_retencion_polizas']*100:>15.2f}%")
print(f"   Prima promedio por póliza:        ${reporte_susc['prima_promedio']:>15,.0f}")
print(f"   Suma asegurada promedio:          ${reporte_susc['suma_asegurada_promedio']:>15,.0f}")
print(f"   % Prima cedida a reaseguro:       {reporte_susc['pct_prima_cedida']*100:>15.2f}%")

In [ ]:
# Visualización del reporte de suscripción
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. Primas por línea de negocio
primas_plot = primas_linea.sort_values('Prima_Emitida', ascending=False)
ax1.barh(primas_plot['Linea'], primas_plot['Prima_Emitida']/1e6, 
         color='#3498db', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Prima Emitida (Millones MXN)', fontsize=11)
ax1.set_title('Prima Emitida por Línea de Negocio', fontsize=12, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

for i, (linea, prima) in enumerate(zip(primas_plot['Linea'], primas_plot['Prima_Emitida'])):
    ax1.text(prima/1e6, i, f'  ${prima/1e6:.1f}M', va='center', 
             fontsize=10, fontweight='bold')

# 2. Distribución de pólizas
polizas_data = [
    reporte_susc['polizas_inicio'],
    reporte_susc['polizas_emitidas'],
    -reporte_susc['polizas_canceladas']
]
polizas_labels = ['Inicio', 'Emitidas', 'Canceladas']
polizas_colors = ['#3498db', '#2ecc71', '#e74c3c']

x_pos = [0, 1, 2]
bars = ax2.bar(x_pos, polizas_data, color=polizas_colors, alpha=0.7, edgecolor='black')
ax2.set_xticks(x_pos)
ax2.set_xticklabels(polizas_labels)
ax2.set_ylabel('Número de Pólizas', fontsize=11)
ax2.set_title('Movimiento de Pólizas en el Trimestre', fontsize=12, fontweight='bold')
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, polizas_data):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
            f'{abs(int(val)):,}', ha='center', 
            va='bottom' if val > 0 else 'top',
            fontsize=10, fontweight='bold')

# 3. Prima emitida vs devengada
categorias_prima = ['Prima\nEmitida', 'Prima\nDevengada', 'Prima\nCedida', 'Prima\nRetenida']
valores_prima = [
    reporte_susc['prima_emitida']/1e6,
    reporte_susc['prima_devengada']/1e6,
    reporte_susc['prima_cedida']/1e6,
    reporte_susc['prima_retenida']/1e6
]
colors_prima = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6']

bars = ax3.bar(categorias_prima, valores_prima, color=colors_prima, 
               alpha=0.7, edgecolor='black')
ax3.set_ylabel('Millones MXN', fontsize=11)
ax3.set_title('Análisis de Primas', fontsize=12, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, valores_prima):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'${val:.1f}M', ha='center', va='bottom',
            fontsize=10, fontweight='bold')

# 4. Número de pólizas por línea
ax4.pie(primas_plot['Num_Polizas'], labels=primas_plot['Linea'], 
        autopct='%1.1f%%', startangle=90,
        colors=['#3498db', '#2ecc71', '#f39c12', '#e74c3c', '#9b59b6'][:len(primas_plot)],
        textprops={'fontsize': 10})
ax4.set_title('Distribución de Pólizas por Línea', fontsize=12, fontweight='bold')

plt.suptitle('Reporte de Suscripción Q3 2024', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 3. Reporte de Siniestros (Siniestralidad y Reservas)

Generemos el reporte de siniestros y reservas técnicas.

In [ ]:
# Cargar datos de siniestros
siniestros_df = pd.read_csv('../examples/data/siniestros_ejemplo.csv')

print("📊 REPORTE TRIMESTRAL DE SINIESTROS")
print("="*90)
print(f"Período: Q3 2024 (Julio - Septiembre)")
print(f"Fecha de generación: {datetime.now().strftime('%Y-%m-%d')}")
print("="*90)

# Generar reporte
generador_sin = GeneradorSiniestros()
reporte_sin = generador_sin.generar(
    siniestros=siniestros_df,
    primas=cartera_df,
    periodo_inicio=date(2024, 7, 1),
    periodo_fin=date(2024, 9, 30)
)

# 1. Siniestros por Línea de Negocio
print(f"\n1. SINIESTROS POR LÍNEA DE NEGOCIO")
print(f"   {'─'*80}")

siniestros_linea = reporte_sin['siniestros_por_linea']
print(siniestros_linea.to_string(index=False))

# 2. Siniestralidad
print(f"\n2. INDICADORES DE SINIESTRALIDAD")
print(f"   {'─'*80}")
print(f"   Siniestros reportados:            {reporte_sin['siniestros_reportados']:>10,}")
print(f"   Siniestros pagados:               {reporte_sin['siniestros_pagados']:>10,}")
print(f"   Siniestros pendientes:            {reporte_sin['siniestros_pendientes']:>10,}")

print(f"\n   Monto pagado:                     ${reporte_sin['monto_pagado']/1000:>15,.0f} (Miles)")
print(f"   Monto pendiente:                  ${reporte_sin['monto_pendiente']/1000:>15,.0f} (Miles)")
print(f"   Monto total:                      ${reporte_sin['monto_total']/1000:>15,.0f} (Miles)")

# 3. Ratios
print(f"\n3. RATIOS DE SINIESTRALIDAD")
print(f"   {'─'*80}")
print(f"   Loss Ratio (Pagado):              {reporte_sin['loss_ratio_pagado']*100:>15.2f}%")
print(f"   Loss Ratio (Incurrido):           {reporte_sin['loss_ratio_incurrido']*100:>15.2f}%")
print(f"   Frecuencia de siniestros:         {reporte_sin['frecuencia_siniestros']*100:>15.2f}%")
print(f"   Severidad promedio:               ${reporte_sin['severidad_promedio']:>15,.0f}")

# 4. Reservas
print(f"\n4. RESERVAS TÉCNICAS")
print(f"   {'─'*80}")
print(f"   Reserva de Siniestros Pendientes: ${reporte_sin['reserva_pendientes']/1000:>15,.0f} (Miles)")
print(f"   Reserva IBNR:                     ${reporte_sin['reserva_ibnr']/1000:>15,.0f} (Miles)")
print(f"   Reserva de Riesgos en Curso:      ${reporte_sin['reserva_riesgos_curso']/1000:>15,.0f} (Miles)")
print(f"   {'─'*80}")
print(f"   TOTAL RESERVAS:                   ${reporte_sin['total_reservas']/1000:>15,.0f} (Miles)")

In [ ]:
# Visualización del reporte de siniestros
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))

# 1. Siniestralidad por línea
sin_plot = siniestros_linea.sort_values('Loss_Ratio', ascending=False)
colors_lr = ['#2ecc71' if lr < 70 else '#f39c12' if lr < 85 else '#e74c3c' 
             for lr in sin_plot['Loss_Ratio']]

bars = ax1.barh(sin_plot['Linea'], sin_plot['Loss_Ratio'], 
                color=colors_lr, alpha=0.7, edgecolor='black')
ax1.set_xlabel('Loss Ratio (%)', fontsize=11)
ax1.set_title('Loss Ratio por Línea de Negocio', fontsize=12, fontweight='bold')
ax1.axvline(x=70, color='green', linestyle='--', alpha=0.5, label='Objetivo 70%')
ax1.axvline(x=85, color='orange', linestyle='--', alpha=0.5, label='Alerta 85%')
ax1.legend(fontsize=9)
ax1.grid(axis='x', alpha=0.3)

for i, (linea, lr) in enumerate(zip(sin_plot['Linea'], sin_plot['Loss_Ratio'])):
    ax1.text(lr, i, f'  {lr:.1f}%', va='center', fontsize=10, fontweight='bold')

# 2. Estado de siniestros
estados = ['Pagados', 'Pendientes']
valores_estados = [
    reporte_sin['siniestros_pagados'],
    reporte_sin['siniestros_pendientes']
]
wedges, texts, autotexts = ax2.pie(valores_estados, labels=estados, 
                                     autopct='%1.1f%%', startangle=90,
                                     colors=['#2ecc71', '#f39c12'],
                                     textprops={'fontsize': 11, 'fontweight': 'bold'})
ax2.set_title('Estado de Siniestros', fontsize=12, fontweight='bold')

# 3. Montos: Pagado vs Pendiente vs IBNR
categorias_monto = ['Pagado', 'Pendiente', 'IBNR']
valores_monto = [
    reporte_sin['monto_pagado']/1e6,
    reporte_sin['monto_pendiente']/1e6,
    reporte_sin['reserva_ibnr']/1e6
]
colors_monto = ['#2ecc71', '#f39c12', '#e74c3c']

bars = ax3.bar(categorias_monto, valores_monto, color=colors_monto, 
               alpha=0.7, edgecolor='black')
ax3.set_ylabel('Millones MXN', fontsize=11)
ax3.set_title('Análisis de Siniestros y Reservas', fontsize=12, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, valores_monto):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'${val:.1f}M', ha='center', va='bottom',
            fontsize=10, fontweight='bold')

# 4. Composición de reservas técnicas
reservas_categorias = ['Pendientes', 'IBNR', 'Riesgos en Curso']
reservas_valores = [
    reporte_sin['reserva_pendientes'],
    reporte_sin['reserva_ibnr'],
    reporte_sin['reserva_riesgos_curso']
]

wedges, texts, autotexts = ax4.pie(reservas_valores, labels=reservas_categorias, 
                                     autopct='%1.1f%%', startangle=90,
                                     colors=['#3498db', '#e74c3c', '#2ecc71'],
                                     textprops={'fontsize': 10})
ax4.set_title('Composición de Reservas Técnicas', fontsize=12, fontweight='bold')

plt.suptitle('Reporte de Siniestros Q3 2024', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 4. Reporte de Inversiones (Portafolio y Riesgo)

Generemos el reporte de inversiones y cumplimiento de límites regulatorios.

In [ ]:
# Cargar datos de inversiones
inversiones_df = pd.read_csv('../examples/data/inversiones_ejemplo.csv')

print("📊 REPORTE TRIMESTRAL DE INVERSIONES")
print("="*90)
print(f"Período: Q3 2024 (Julio - Septiembre)")
print(f"Fecha de generación: {datetime.now().strftime('%Y-%m-%d')}")
print("="*90)

# Generar reporte
generador_inv = GeneradorInversiones()
reporte_inv = generador_inv.generar(
    inversiones=inversiones_df,
    fecha_corte=date(2024, 9, 30)
)

# 1. Portafolio por Tipo de Instrumento
print(f"\n1. PORTAFOLIO POR TIPO DE INSTRUMENTO")
print(f"   {'─'*80}")

portafolio = reporte_inv['portafolio_por_tipo']
print(portafolio.to_string(index=False))

# 2. Calificaciones Crediticias
print(f"\n2. CALIFICACIONES CREDITICIAS")
print(f"   {'─'*80}")

calificaciones = reporte_inv['portafolio_por_calificacion']
print(calificaciones.to_string(index=False))

# 3. Indicadores de Riesgo
print(f"\n3. INDICADORES DE RIESGO")
print(f"   {'─'*80}")
print(f"   Valor total del portafolio:       ${reporte_inv['valor_total']/1e6:>15,.2f} Millones")
print(f"   Duración promedio (años):         {reporte_inv['duracion_promedio']:>15.2f}")
print(f"   Rendimiento promedio:             {reporte_inv['rendimiento_promedio']*100:>15.2f}%")
print(f"   Rendimiento YTD:                  {reporte_inv['rendimiento_ytd']*100:>15.2f}%")
print(f"   Volatilidad del portafolio:       {reporte_inv['volatilidad']*100:>15.2f}%")

# 4. Cumplimiento de Límites Regulatorios
print(f"\n4. CUMPLIMIENTO DE LÍMITES REGULATORIOS")
print(f"   {'─'*80}")

limites = reporte_inv['cumplimiento_limites']
for limite in limites:
    status = '✅' if limite['cumple'] else '❌'
    print(f"   {status} {limite['concepto']:30s}: {limite['actual']*100:>6.2f}% / "
          f"{limite['limite']*100:>6.2f}% (límite)")

# 5. Vencimientos
print(f"\n5. PERFIL DE VENCIMIENTOS")
print(f"   {'─'*80}")

vencimientos = reporte_inv['vencimientos']
print(vencimientos.to_string(index=False))

In [ ]:
# Visualización del reporte de inversiones
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. Portafolio por tipo de instrumento
ax1 = fig.add_subplot(gs[0, :2])
port_plot = portafolio.sort_values('Valor', ascending=False)
ax1.barh(port_plot['Tipo'], port_plot['Valor']/1e6, 
         color='#3498db', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Valor (Millones MXN)', fontsize=11)
ax1.set_title('Portafolio por Tipo de Instrumento', fontsize=12, fontweight='bold')
ax1.grid(axis='x', alpha=0.3)

for i, (tipo, val, pct) in enumerate(zip(port_plot['Tipo'], 
                                          port_plot['Valor'], 
                                          port_plot['Porcentaje'])):
    ax1.text(val/1e6, i, f'  ${val/1e6:.1f}M ({pct:.1f}%)', 
             va='center', fontsize=9, fontweight='bold')

# 2. Calificaciones crediticias
ax2 = fig.add_subplot(gs[0, 2])
cal_plot = calificaciones.sort_values('Porcentaje', ascending=False)
colors_cal = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c'][:len(cal_plot)]
wedges, texts, autotexts = ax2.pie(cal_plot['Porcentaje'], 
                                     labels=cal_plot['Calificacion'], 
                                     autopct='%1.1f%%', startangle=90,
                                     colors=colors_cal,
                                     textprops={'fontsize': 9})
ax2.set_title('Calificaciones Crediticias', fontsize=12, fontweight='bold')

# 3. Perfil de vencimientos
ax3 = fig.add_subplot(gs[1, :])
bars = ax3.bar(vencimientos['Plazo'], vencimientos['Valor']/1e6, 
               color='#2ecc71', alpha=0.7, edgecolor='black')
ax3.set_ylabel('Valor (Millones MXN)', fontsize=11)
ax3.set_xlabel('Plazo de Vencimiento', fontsize=11)
ax3.set_title('Perfil de Vencimientos del Portafolio', fontsize=12, fontweight='bold')
ax3.grid(axis='y', alpha=0.3)

for bar, val, pct in zip(bars, vencimientos['Valor'], vencimientos['Porcentaje']):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height,
            f'${val/1e6:.1f}M\n({pct:.1f}%)',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

# 4. Cumplimiento de límites
ax4 = fig.add_subplot(gs[2, :2])
conceptos_lim = [l['concepto'] for l in limites]
actual_lim = [l['actual']*100 for l in limites]
limite_lim = [l['limite']*100 for l in limites]
cumple_lim = [l['cumple'] for l in limites]

x = np.arange(len(conceptos_lim))
width = 0.35

colors_cumple = ['#2ecc71' if c else '#e74c3c' for c in cumple_lim]
bars1 = ax4.bar(x - width/2, actual_lim, width, label='Actual', 
                color=colors_cumple, alpha=0.7, edgecolor='black')
bars2 = ax4.bar(x + width/2, limite_lim, width, label='Límite', 
                color='#95a5a6', alpha=0.5, edgecolor='black')

ax4.set_ylabel('Porcentaje (%)', fontsize=11)
ax4.set_title('Cumplimiento de Límites Regulatorios', fontsize=12, fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels([c[:20] + '...' if len(c) > 20 else c for c in conceptos_lim], 
                     rotation=15, ha='right')
ax4.legend(fontsize=10)
ax4.grid(axis='y', alpha=0.3)

# 5. Métricas de riesgo
ax5 = fig.add_subplot(gs[2, 2])
ax5.axis('off')

metricas_texto = f"""
MÉTRICAS DE RIESGO
{'─'*30}

Valor Total:
${reporte_inv['valor_total']/1e6:.2f} Millones

Duración Promedio:
{reporte_inv['duracion_promedio']:.2f} años

Rendimiento Promedio:
{reporte_inv['rendimiento_promedio']*100:.2f}%

Rendimiento YTD:
{reporte_inv['rendimiento_ytd']*100:.2f}%

Volatilidad:
{reporte_inv['volatilidad']*100:.2f}%
"""

ax5.text(0.1, 0.95, metricas_texto, transform=ax5.transAxes,
         fontsize=11, verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Reporte de Inversiones Q3 2024', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 5. Reporte RCS (Capital Regulatorio)

Generemos el reporte de capital y solvencia.

In [ ]:
print("📊 REPORTE TRIMESTRAL DE CAPITAL Y SOLVENCIA (RCS)")
print("="*90)
print(f"Período: Q3 2024 (al 30 de Septiembre)")
print(f"Fecha de generación: {datetime.now().strftime('%Y-%m-%d')}")
print("="*90)

# Generar reporte RCS
generador_rcs = GeneradorRCS()
reporte_rcs = generador_rcs.generar(
    capital_disponible=Decimal('200000000'),  # 200M
    reservas_tecnicas=reporte_sin['total_reservas'],
    primas_emitidas=reporte_susc['prima_emitida'],
    siniestros_incurridos=reporte_sin['monto_total'],
    inversiones=reporte_inv['valor_total']
)

# 1. Capital Disponible
print(f"\n1. CAPITAL DISPONIBLE")
print(f"   {'─'*80}")

capital_comp = reporte_rcs['capital_componentes']
print(capital_comp.to_string(index=False))

print(f"\n   {'─'*80}")
print(f"   CAPITAL TOTAL DISPONIBLE:         ${reporte_rcs['capital_total']/1e6:>15,.2f} Millones")

# 2. RCS por Componente
print(f"\n2. REQUERIMIENTO DE CAPITAL DE SOLVENCIA (RCS)")
print(f"   {'─'*80}")

rcs_comp = reporte_rcs['rcs_componentes']
print(rcs_comp.to_string(index=False))

print(f"\n   {'─'*80}")
print(f"   RCS TOTAL (agregado):             ${reporte_rcs['rcs_total']/1e6:>15,.2f} Millones")

# 3. Ratio de Solvencia
print(f"\n3. RATIO DE SOLVENCIA")
print(f"   {'─'*80}")
print(f"   Capital Disponible:               ${reporte_rcs['capital_total']/1e6:>15,.2f} Millones")
print(f"   RCS Requerido:                    ${reporte_rcs['rcs_total']/1e6:>15,.2f} Millones")
print(f"   {'─'*80}")
print(f"   RATIO DE SOLVENCIA:               {reporte_rcs['ratio_solvencia']*100:>15,.2f}%")
print(f"   {'─'*80}")

# Clasificación
ratio = reporte_rcs['ratio_solvencia'] * 100
if ratio >= 150:
    status = "✅ SOLVENTE - Excedente significativo"
    color = "verde"
elif ratio >= 100:
    status = "⚠️ ADECUADO - Vigilancia"
    color = "amarillo"
else:
    status = "❌ INSUFICIENTE - Plan de regularización requerido"
    color = "rojo"

print(f"   CLASIFICACIÓN: {status}")

# 4. Excedente/Déficit
excedente = reporte_rcs['capital_total'] - reporte_rcs['rcs_total']
print(f"\n4. EXCEDENTE/DÉFICIT DE CAPITAL")
print(f"   {'─'*80}")
if excedente > 0:
    print(f"   ✅ Excedente de capital:          ${excedente/1e6:>15,.2f} Millones")
    print(f"   Margen adicional sobre 100%:      {(ratio - 100):>15,.2f} puntos porcentuales")
else:
    print(f"   ❌ Déficit de capital:            ${abs(excedente)/1e6:>15,.2f} Millones")
    print(f"   Capital adicional requerido:      ${abs(excedente)/1e6:>15,.2f} Millones")

In [ ]:
# Visualización del reporte RCS
fig = plt.figure(figsize=(18, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# 1. Gauge de ratio de solvencia (grande)
ax1 = fig.add_subplot(gs[0, :], projection='polar')

# Crear gauge
theta_ranges = [0, np.pi/3, 2*np.pi/3, np.pi]
colors_gauge = ['#e74c3c', '#f39c12', '#2ecc71']

for i in range(len(theta_ranges)-1):
    theta_section = np.linspace(theta_ranges[i], theta_ranges[i+1], 50)
    ax1.fill_between(theta_section, 0, 1, color=colors_gauge[i], alpha=0.3)

# Aguja
ratio_val = float(reporte_rcs['ratio_solvencia'] * 100)
ratio_normalized = min(ratio_val / 200, 1.0)
angle = np.pi * (1 - ratio_normalized)
ax1.plot([angle, angle], [0, 0.9], 'k-', linewidth=5)
ax1.plot(angle, 0.9, 'ko', markersize=20)

ax1.set_ylim(0, 1)
ax1.set_theta_zero_location('W')
ax1.set_theta_direction(1)
ax1.set_xticks([0, np.pi/3, 2*np.pi/3, np.pi])
ax1.set_xticklabels(['200%', '150%', '100%', '0%'], fontsize=12)
ax1.set_yticks([])
ax1.set_title(f'RATIO DE SOLVENCIA: {ratio_val:.1f}%\n{status}', 
              fontsize=16, fontweight='bold', pad=30)

# 2. Composición del capital
ax2 = fig.add_subplot(gs[1, 0])
capital_tipos = capital_comp['Componente'].tolist()
capital_valores = capital_comp['Monto'].tolist()
colors_cap = ['#3498db', '#2ecc71', '#f39c12', '#9b59b6'][:len(capital_tipos)]

wedges, texts, autotexts = ax2.pie(capital_valores, labels=capital_tipos, 
                                     autopct='%1.1f%%', startangle=90,
                                     colors=colors_cap,
                                     textprops={'fontsize': 9})
ax2.set_title('Composición del Capital', fontsize=12, fontweight='bold')

# 3. Componentes del RCS
ax3 = fig.add_subplot(gs[1, 1])
rcs_tipos = rcs_comp['Riesgo'].tolist()
rcs_valores = [v/1e6 for v in rcs_comp['Monto'].tolist()]

bars = ax3.barh(rcs_tipos, rcs_valores, color='#e74c3c', alpha=0.7, edgecolor='black')
ax3.set_xlabel('Millones MXN', fontsize=11)
ax3.set_title('Componentes del RCS', fontsize=12, fontweight='bold')
ax3.grid(axis='x', alpha=0.3)

for i, (tipo, val) in enumerate(zip(rcs_tipos, rcs_valores)):
    ax3.text(val, i, f'  ${val:.1f}M', va='center', fontsize=9, fontweight='bold')

# 4. Capital vs RCS
ax4 = fig.add_subplot(gs[1, 2])
categorias_vs = ['Capital\nDisponible', 'RCS\nRequerido']
valores_vs = [
    float(reporte_rcs['capital_total'])/1e6,
    float(reporte_rcs['rcs_total'])/1e6
]
colors_vs = ['#2ecc71', '#e74c3c']

bars = ax4.bar(categorias_vs, valores_vs, color=colors_vs, 
               alpha=0.7, edgecolor='black', linewidth=2)
ax4.set_ylabel('Millones MXN', fontsize=11)
ax4.set_title('Capital Disponible vs RCS', fontsize=12, fontweight='bold')
ax4.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, valores_vs):
    height = bar.get_height()
    ax4.text(bar.get_x() + bar.get_width()/2., height,
            f'${val:.1f}M', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

# Línea del excedente
if excedente > 0:
    ax4.plot([0.5], [float(reporte_rcs['rcs_total'])/1e6 + float(excedente)/1e6/2], 
             marker='v', markersize=15, color='green')
    ax4.text(0.5, float(reporte_rcs['rcs_total'])/1e6 + float(excedente)/1e6/2,
            f'  Excedente\n  ${float(excedente)/1e6:.1f}M',
            ha='left', va='center', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.7))

plt.suptitle('Reporte de Capital y Solvencia (RCS) Q3 2024', 
             fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

## 6. Exportación a Excel y PDF

Exportemos todos los reportes a formatos utilizables.

In [ ]:
# Exportar a Excel
print("📁 EXPORTACIÓN DE REPORTES")
print("="*90)

exportador = ExportadorExcel()

# Crear diccionario con todos los reportes
reportes_excel = {
    'Suscripcion': {
        'Primas por Linea': reporte_susc['primas_por_linea'],
        'Resumen': pd.DataFrame([
            {'Concepto': 'Prima Emitida', 'Valor': reporte_susc['prima_emitida']},
            {'Concepto': 'Prima Devengada', 'Valor': reporte_susc['prima_devengada']},
            {'Concepto': 'Prima Cedida', 'Valor': reporte_susc['prima_cedida']},
            {'Concepto': 'Prima Retenida', 'Valor': reporte_susc['prima_retenida']},
            {'Concepto': 'Polizas Inicio', 'Valor': reporte_susc['polizas_inicio']},
            {'Concepto': 'Polizas Fin', 'Valor': reporte_susc['polizas_fin']},
        ])
    },
    'Siniestros': {
        'Siniestros por Linea': reporte_sin['siniestros_por_linea'],
        'Resumen': pd.DataFrame([
            {'Concepto': 'Siniestros Reportados', 'Valor': reporte_sin['siniestros_reportados']},
            {'Concepto': 'Siniestros Pagados', 'Valor': reporte_sin['siniestros_pagados']},
            {'Concepto': 'Monto Pagado', 'Valor': reporte_sin['monto_pagado']},
            {'Concepto': 'Loss Ratio Incurrido', 'Valor': reporte_sin['loss_ratio_incurrido']},
            {'Concepto': 'Total Reservas', 'Valor': reporte_sin['total_reservas']},
        ])
    },
    'Inversiones': {
        'Portafolio': reporte_inv['portafolio_por_tipo'],
        'Calificaciones': reporte_inv['portafolio_por_calificacion'],
        'Vencimientos': reporte_inv['vencimientos'],
        'Metricas': pd.DataFrame([
            {'Metrica': 'Valor Total', 'Valor': reporte_inv['valor_total']},
            {'Metrica': 'Duracion Promedio', 'Valor': reporte_inv['duracion_promedio']},
            {'Metrica': 'Rendimiento Promedio', 'Valor': reporte_inv['rendimiento_promedio']},
            {'Metrica': 'Rendimiento YTD', 'Valor': reporte_inv['rendimiento_ytd']},
            {'Metrica': 'Volatilidad', 'Valor': reporte_inv['volatilidad']},
        ])
    },
    'RCS': {
        'Capital': reporte_rcs['capital_componentes'],
        'RCS Componentes': reporte_rcs['rcs_componentes'],
        'Resumen': pd.DataFrame([
            {'Concepto': 'Capital Total', 'Valor': reporte_rcs['capital_total']},
            {'Concepto': 'RCS Total', 'Valor': reporte_rcs['rcs_total']},
            {'Concepto': 'Ratio Solvencia (%)', 'Valor': reporte_rcs['ratio_solvencia'] * 100},
        ])
    }
}

# Exportar
archivo_excel = '../examples/reportes_cnsf_q3_2024.xlsx'
resultado_export = exportador.exportar(
    reportes=reportes_excel,
    archivo_destino=archivo_excel,
    incluir_graficas=True
)

print(f"\n✅ Reportes exportados exitosamente:")
print(f"   Archivo Excel: {archivo_excel}")
print(f"   Hojas creadas: {len(reportes_excel)}")
print(f"   Tamaño: {resultado_export['tamaño_kb']:.2f} KB")

# Resumen de contenido
print(f"\n📋 Contenido del archivo Excel:")
for hoja, contenido in reportes_excel.items():
    print(f"   - {hoja}: {len(contenido)} tablas")
    for tabla_nombre in contenido.keys():
        print(f"     • {tabla_nombre}")

print(f"\n💡 El archivo puede ser abierto en Excel para análisis adicional")
print(f"   y entrega a la CNSF.")

In [ ]:
# Resumen ejecutivo
print("\n" + "="*90)
print("RESUMEN EJECUTIVO - REPORTES CNSF Q3 2024")
print("="*90)

print(f"\n📊 SUSCRIPCIÓN")
print(f"   Prima Emitida:      ${reporte_susc['prima_emitida']/1e6:>10,.2f} Millones")
print(f"   Pólizas en Vigor:   {reporte_susc['polizas_fin']:>10,}")
print(f"   Retención:          {(1-reporte_susc['pct_prima_cedida'])*100:>10,.1f}%")

print(f"\n💰 SINIESTROS")
print(f"   Loss Ratio:         {reporte_sin['loss_ratio_incurrido']*100:>10,.1f}%")
print(f"   Reservas Totales:   ${reporte_sin['total_reservas']/1e6:>10,.2f} Millones")
print(f"   Siniestros Pagados: {reporte_sin['siniestros_pagados']:>10,}")

print(f"\n📈 INVERSIONES")
print(f"   Valor Portafolio:   ${reporte_inv['valor_total']/1e6:>10,.2f} Millones")
print(f"   Rendimiento YTD:    {reporte_inv['rendimiento_ytd']*100:>10,.2f}%")
print(f"   Duración:           {reporte_inv['duracion_promedio']:>10,.2f} años")

print(f"\n💪 CAPITAL Y SOLVENCIA")
print(f"   Capital Disponible: ${reporte_rcs['capital_total']/1e6:>10,.2f} Millones")
print(f"   RCS Requerido:      ${reporte_rcs['rcs_total']/1e6:>10,.2f} Millones")
print(f"   Ratio Solvencia:    {reporte_rcs['ratio_solvencia']*100:>10,.1f}%  {status}")

print(f"\n" + "="*90)
print(f"Reportes generados: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"{'='*90}\n")

## Conclusiones

En este notebook aprendimos a:

1. **Reporte de Suscripción**:
   - Primas por línea de negocio
   - Movimiento de pólizas
   - Indicadores de retención y prima promedio

2. **Reporte de Siniestros**:
   - Siniestralidad por línea
   - Loss ratios (pagado e incurrido)
   - Reservas técnicas (pendientes, IBNR, riesgos en curso)

3. **Reporte de Inversiones**:
   - Composición del portafolio
   - Calificaciones crediticias
   - Cumplimiento de límites regulatorios
   - Perfil de vencimientos

4. **Reporte RCS**:
   - Capital disponible por componente
   - RCS por tipo de riesgo
   - Ratio de solvencia
   - Excedente/déficit de capital

5. **Exportación**:
   - Generación de archivos Excel
   - Múltiples hojas organizadas
   - Listo para entrega a CNSF

### Beneficios de la Automatización:
- ✅ Consistencia en reportes
- ✅ Reducción de errores manuales
- ✅ Ahorro de tiempo (horas → minutos)
- ✅ Trazabilidad y auditoría
- ✅ Cumplimiento regulatorio

### Próximos Pasos

Continúa con:
- **07_casos_practicos_completos.ipynb**: Workflows end-to-end integrando todo el proceso
  desde tarificación hasta reportes regulatorios